# Analise de Tabuleiros de Xadrez
## Visao computacional classica para leitura de tabuleiros, identificacao e classificacao de pecas

**Disciplina:** INE410121 / TRV410001 — Visao Computacional - UFSC
**Dataset:** Synthetic Chess Board Images — Kaggle (thefamousrat)

# Introducao

Este notebook apresenta uma **pipeline completa de Visao Computacional Classica** aplicada ao problema de deteccao e analise de tabuleiros de xadrez. O objetivo e demonstrar como tecnicas tradicionais de processamento de imagens podem ser combinadas para:

1. **Detectar** um tabuleiro de xadrez em uma imagem;
2. **Corrigir a perspectiva** para obter uma visao top-down;
3. **Segmentar** o tabuleiro em um grid 8x8;
4. **Identificar** quais casas estao ocupadas por pecas;
5. **Detectar jogadas** comparando dois estados do tabuleiro.

A escolha de um cenario de xadrez e motivada pela riqueza de tecnicas classicas que o problema exige: deteccao de bordas, transformada de Hough para linhas, deteccao de contornos, correcao de perspectiva via homografia, limiarizacao, operacoes morfologicas e analise de features visuais. **Nenhuma tecnica de Deep Learning ou modelo pre-treinado e utilizada.**

# Definicao do Problema

**Dado**: Uma imagem renderizada de um tabuleiro de xadrez com pecas, fotografada em perspectiva angular.

**Objetivo**: Determinar automaticamente o estado do tabuleiro — quais casas (A1–H8) estao ocupadas — usando exclusivamente tecnicas classicas de visao computacional.

## Aplicacoes reais

- **Digitalizacao de partidas**: converter jogos fisicos em notacao digital (PGN);
- **Assistentes de analise**: alimentar engines com posicao detectada por camera;
- **Acessibilidade**: descricao automatica de partidas;
- **Ensino de CV**: tabuleiro e um objeto rico em geometria regular.

## Desafios

- **Perspectiva**: camera em angulo causa distorcao projetiva;
- **Iluminacao**: sombras e reflexos alteram aparencia;
- **Similaridade visual**: pecas de madeira sobre tabuleiro de madeira criam baixo contraste peca-fundo — este e o principal desafio para abordagens classicas.

# Descricao do Dataset

**Synthetic Chess Board Images** ([Kaggle — thefamousrat](https://www.kaggle.com/datasets/thefamousrat/synthetic-chess-board-images))

Imagens sinteticas fotorrealistas de tabuleiros de xadrez geradas com Blender Cycles.

| Propriedade | Valor |
|---|---|
| Resolucao | 1280 x 1280 pixels (JPEG) |
| Perspectiva | Vista angular (nao top-down) |
| Renderizacao | Blender Cycles (fotorrealista) |
| Pecas | Madeira (tons de marrom) |
| Tabuleiro | Madeira (tons de marrom) |
| Licenca | CC0 — Dominio Publico |

Cada imagem `X.jpg` possui anotacao `X.json`:
- **config**: posicoes das pecas (ex: `{"A1": "pawn_w", "E4": "queen_b"}`);
- **corners**: 4 cantos do tabuleiro em coordenadas normalizadas [0, 1].

> **Nota sobre o dataset**: As pecas e o tabuleiro sao feitos do mesmo material (madeira), resultando em baixo contraste visual entre peca e fundo. Isso torna a deteccao de ocupacao particularmente desafiadora para metodos classicos, mas e excelente para demonstrar os limites e potencialidades dessas tecnicas.

# 1. Configuracao do Ambiente

Configuracao automatica que funciona tanto **localmente** quanto no **Google Colab**.
Localmente, usa o `sys.path` para encontrar o modulo `src/`.
No Colab, clona o repositorio e instala as dependencias automaticamente.

O dataset e baixado via `kagglehub` na primeira execucao (~457 MB).

In [ ]:
import sys, os

# --- Detecta ambiente: Colab ou local ---
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Clonar o repositorio
    if not os.path.exists("classical-cv"):
        !git clone https://github.com/daviludvig/classical-cv.git
    os.chdir("classical-cv")

    # 2. Instalar dependencias
    !pip install -q opencv-python-headless kagglehub python-dotenv ipywidgets

    # 3. Configurar credenciais do Kaggle (preencha com seu token)
    os.environ["KAGGLE_API_TOKEN"] = "KGAT_SEU_TOKEN_AQUI"  # <-- EDITAR
else:
    sys.path.insert(0, os.path.abspath(".."))

from src.setup import setup
DATA_DIR = setup()

## 1.2. Importacoes e configuracoes globais

Importamos as bibliotecas principais e todas as funcoes do modulo `src/chess.py`:
- **OpenCV** (`cv2`): processamento de imagem, deteccao de bordas, homografia
- **NumPy**: manipulacao de arrays e matrizes
- **Matplotlib**: visualizacao de resultados

O diretorio `RUN_DIR` armazena as figuras geradas nesta execucao com timestamp unico.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

from src.chess import (
    list_samples, load_annotation, annotation_path_for, corners_to_pixels,
    ground_truth_occupancy, cell_name, square_to_grid,
    to_gray, blur, adaptive_threshold, morph_clean,
    detect_edges, detect_lines_hough, classify_lines,
    find_board_contour, order_corners, compute_homography, warp_board,
    segment_grid, cell_features, is_occupied, build_occupancy_map,
    detect_moves, _resolve_data_dir,
    draw_corners, draw_grid_overlay, draw_occupancy_image,
    draw_move_diff, draw_hough_lines, detect_grid_in_warped, grid_corners_from_warped,
    detect_board_corners, frame_to_grid_corners, FRAME_BORDER_FRAC,
    detect_board_rotation, best_rotation_vs_gt,
    detect_edges_roberts,
)
from src.utils import create_run_dir, save_fig

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = False

RUN_DIR = create_run_dir()

# 2. Carregamento e Visualizacao das Imagens

Carregamos o dataset e exploramos visualmente as imagens.
Cada imagem e um render fotorrealista (Blender Cycles) de um tabuleiro de xadrez em perspectiva angular.

**Parametros**:
- `DATASET`: caminho para o diretorio do dataset (gerenciado por `kagglehub`)

In [ ]:
DATASET = Path(DATA_DIR)
images = list_samples(DATASET)
print(f"Total de imagens no dataset: {len(images)}")

DATA_ROOT = _resolve_data_dir(DATASET)
config_path = DATA_ROOT / "config.json"
if config_path.exists():
    with open(config_path) as f:
        dataset_config = json.load(f)
    print(f"Tipos de pecas: {dataset_config.get('piecesTypes', 'N/A')}")

## 2.2. Amostra de imagens do dataset

Mostramos as primeiras imagens para entender as caracteristicas visuais do dataset:
resolucao, iluminacao, angulo de camera, e variedade de posicoes das pecas.

**Parametros**:
- `N_SHOW`: quantidade de imagens a exibir (padrao: 6)

In [ ]:
N_SHOW = min(6, len(images))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, images[:N_SHOW]):
    img = cv2.imread(str(img_path))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.stem, fontsize=10)
    ax.axis("off")
plt.suptitle("Amostra de imagens do dataset", fontsize=14)
plt.tight_layout()
save_fig("01_dataset_samples", RUN_DIR)
plt.show()

## 2.3. Anotacoes do dataset

Cada imagem possui um arquivo JSON com:
- **config**: casas ocupadas → tipo de peca (ex: `"A3": "knight_b"`);
- **corners**: 4 coordenadas normalizadas [0, 1] dos cantos do tabuleiro.

As coordenadas normalizadas sao convertidas para pixels multiplicando pelas dimensoes da imagem.

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

def explore_corners(ref_idx, border_frac):
    global ref_img, ref_ann, ref_corners, DST_SIZE
    DST_SIZE = 480

    ref_path = images[ref_idx]
    ref_img = cv2.imread(str(ref_path))
    ref_ann = load_annotation(annotation_path_for(ref_path))

    raw_corners = order_corners(corners_to_pixels(ref_ann["corners"], ref_img.shape))

    if border_frac > 0:
        ref_corners = frame_to_grid_corners(raw_corners, border_frac)
    else:
        ref_corners = raw_corners

    vis_overlay = draw_grid_overlay(ref_img, ref_corners, color=(0, 255, 0), thickness=2)
    vis_overlay = draw_corners(vis_overlay, ref_corners)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(vis_overlay, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Grid overlay (border_frac={border_frac:.3f})")
    axes[0].axis("off")

    H = compute_homography(ref_corners, dst_size=DST_SIZE, border_frac=0)
    warped = warp_board(ref_img, H, size=DST_SIZE)
    wv = warped.copy()
    cs = DST_SIZE // 8
    for k in range(9):
        cv2.line(wv, (k*cs, 0), (k*cs, DST_SIZE), (0, 255, 0), 2)
        cv2.line(wv, (0, k*cs), (DST_SIZE, k*cs), (0, 255, 0), 2)
    axes[1].imshow(cv2.cvtColor(wv, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Warped + grid ({DST_SIZE}x{DST_SIZE})")
    axes[1].axis("off")

    plt.suptitle(f"Imagem {ref_path.stem} — ajuste de cantos", fontsize=13)
    plt.tight_layout()
    plt.show()

    print(f"Imagem: {images[ref_idx].name} | Casas ocupadas: {len(ref_ann['config'])}")

interact(
    explore_corners,
    ref_idx=IntSlider(value=0, min=0, max=min(49, len(images)-1), step=1,
                      description="Imagem:"),
    border_frac=FloatSlider(value=0.0, min=-0.05, max=0.05, step=0.005,
                            description="border_frac:", readout_format=".3f"),
);

# 3. Deteccao do Tabuleiro

Demonstramos tres tecnicas classicas:
1. **Deteccao de bordas** (Canny) — encontra descontinuidades de intensidade;
2. **Deteccao de linhas** (Transformada de Hough) — encontra linhas nas bordas;
3. **Deteccao de contornos** — encontra o contorno quadrilateral do tabuleiro.

## 3.1. Conversao para tons de cinza e suavizacao gaussiana

**Por que**: A maioria dos operadores de bordas (Canny, Sobel, Roberts) opera sobre imagens em tons de cinza. A suavizacao gaussiana reduz ruido de alta frequencia antes da deteccao de bordas, evitando falsos positivos.

**Conceito**: Filtragem no **dominio espacial** — convolucao da imagem com um kernel gaussiano 2D. O parametro `ksize` controla o tamanho do kernel e, consequentemente, o grau de suavizacao.

**Parametros**:
- `ksize=5`: kernel gaussiano 5x5 (valores maiores = mais suavizacao)

In [ ]:
gray = to_gray(ref_img)
blurred = blur(gray, ksize=5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (BGR)")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Tons de cinza")
axes[2].imshow(blurred, cmap="gray")
axes[2].set_title("Gaussiana (k=5)")
for ax in axes:
    ax.axis("off")
plt.suptitle("Pre-processamento: conversao e suavizacao", fontsize=13)
plt.tight_layout()
save_fig("03_preprocessing", RUN_DIR)
plt.show()

## 3.1.1. Histograma e equalizacao

**Conceito**: O histograma mostra a distribuicao dos niveis de cinza da imagem.
A **equalizacao de histograma** redistribui as intensidades para aumentar o contraste.
O **CLAHE** (Contrast-Limited Adaptive Histogram Equalization) faz a equalizacao localmente,
preservando detalhes sem saturar regioes uniformes.


In [ ]:
from src.chess import equalize_histogram, clahe

eq = equalize_histogram(gray)
cl = clahe(gray, clip=2.0, grid=8)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, img_g, title in zip(axes[0], [gray, eq, cl], ["Original", "Eq. Global", "CLAHE"]):
    ax.imshow(img_g, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
for ax, img_g, title in zip(axes[1], [gray, eq, cl], ["Original", "Eq. Global", "CLAHE"]):
    ax.hist(img_g.ravel(), bins=256, range=(0, 256), color="steelblue", alpha=0.7)
    ax.set_xlim(0, 256)
    ax.set_title(f"Histograma — {title}")
    ax.set_ylabel("Pixels")
plt.suptitle("Analise de histograma e equalizacao de contraste", fontsize=13)
plt.tight_layout()
save_fig("03b_histogram_equalization", RUN_DIR)
plt.show()


## 3.2. Deteccao de bordas — Canny

**Por que**: Bordas sao a base para detectar as linhas do grid do tabuleiro. O operador de Canny e o mais robusto entre os classicos, combinando multiplas etapas de filtragem.

**Conceito**: O operador de Canny combina quatro etapas:
1. Suavizacao gaussiana (ja aplicada)
2. Calculo do gradiente via **Sobel** (magnitude e direcao)
3. **Supressao de nao-maximos**: afina bordas a 1 pixel de largura
4. **Limiarizacao por histerese**: pixels com gradiente acima de `high` sao borda; entre `low` e `high` sao borda apenas se conectados a uma borda forte.

**Parametros testados** (low, high):
- `(30, 80)`: sensivel — detecta muitas bordas, inclusive ruido
- `(50, 150)`: equilibrado — bom compromisso (usado no restante do pipeline)
- `(80, 200)`: conservador — apenas bordas fortes

In [ ]:
params = [(30, 80), (50, 150), (80, 200)]
fig, axes = plt.subplots(1, len(params) + 1, figsize=(16, 4))
axes[0].imshow(blurred, cmap="gray")
axes[0].set_title("Entrada (blur)")
axes[0].axis("off")

for ax, (low, high) in zip(axes[1:], params):
    edges = detect_edges(blurred, low, high)
    ax.imshow(edges, cmap="gray")
    ax.set_title(f"Canny ({low}, {high})")
    ax.axis("off")

plt.suptitle("Deteccao de bordas — efeito dos limiares Canny", fontsize=13)
plt.tight_layout()
save_fig("04_canny_edges", RUN_DIR)
plt.show()

edges = detect_edges(blurred, 50, 150)

## 3.2.1. Comparacao de operadores de borda

Diferentes operadores de borda usam kernels distintos para calcular gradientes:

| Operador | Kernel | Caracteristicas |
|---|---|---|
| **Roberts** | 2x2 cruzado | Simples, sensivel a ruido |
| **Prewitt** | 3x3 media | Suavizacao uniforme |
| **Sobel** | 3x3 ponderado | Mais peso no centro, robusto |
| **Canny** | Multi-etapa | Supressao nao-maxima + histerese |


In [ ]:
from src.chess import detect_edges_roberts, detect_edges_prewitt, detect_edges_sobel

edge_roberts = detect_edges_roberts(blurred, thresh=30)
edge_prewitt = detect_edges_prewitt(blurred, thresh=30)
edge_sobel = detect_edges_sobel(blurred, thresh=30)
edge_canny = detect_edges(blurred, low=50, high=150)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, e, name in zip(axes, [edge_roberts, edge_prewitt, edge_sobel, edge_canny],
                        ["Roberts", "Prewitt", "Sobel", "Canny (50,150)"]):
    ax.imshow(e, cmap="gray")
    ax.set_title(f"{name}\n{np.count_nonzero(e)} px")
    ax.axis("off")
plt.suptitle("Comparacao de operadores de deteccao de bordas", fontsize=13)
plt.tight_layout()
save_fig("04b_edge_operators_comparison", RUN_DIR)
plt.show()


## 3.2.2. Operacoes morfologicas

**Conceito**: Operacoes sobre a forma dos objetos binarios:
- **Erosao**: remove pixels na borda → elimina ruido pequeno
- **Dilatacao**: expande bordas → conecta regioes proximas
- **Abertura** (erosao + dilatacao): remove ruido preservando tamanho
- **Fechamento** (dilatacao + erosao): preenche buracos preservando tamanho


In [ ]:
kernel_3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
kernel_5 = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))

eroded = cv2.erode(edges, kernel_3, iterations=1)
dilated = cv2.dilate(edges, kernel_3, iterations=1)
opened = cv2.morphologyEx(edges, cv2.MORPH_OPEN, kernel_5)
closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel_5)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, img_b, title in zip(axes,
    [edges, eroded, dilated, opened, closed],
    ["Original (Canny)", "Erosao 3x3", "Dilatacao 3x3", "Abertura 5x5", "Fechamento 5x5"]):
    ax.imshow(img_b, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle("Operacoes morfologicas sobre bordas Canny", fontsize=13)
plt.tight_layout()
save_fig("04c_morphological_operations", RUN_DIR)
plt.show()


## 3.3. Deteccao de linhas — Transformada de Hough

**Por que**: As linhas do grid do tabuleiro sao o elemento geometrico fundamental. A Transformada de Hough converte o problema de detectar linhas no espaco da imagem em um problema de detectar **picos** no espaco parametrico.

**Conceito**: Cada pixel de borda $(x, y)$ "vota" por todas as linhas $(\rho, \theta)$ que passam por ele no espaco parametrico. Acumulacoes de votos acima do `threshold` indicam linhas reais. Classificamos as linhas detectadas em:
- **Horizontais**: $\theta \approx \pi/2$ (linhas das fileiras)
- **Verticais**: $\theta \approx 0$ (linhas das colunas)

**Parametros**:
- `threshold=100`: minimo de votos para considerar uma linha

In [ ]:
lines = detect_lines_hough(edges, threshold=100)
print(f"Linhas detectadas: {len(lines) if lines is not None else 0}")

vis_lines = draw_hough_lines(ref_img, lines, color=(0, 0, 255), thickness=1)
h_lines, v_lines = classify_lines(lines)
print(f"Horizontais: {len(h_lines)}, Verticais: {len(v_lines)}")

# Desenhar H (verde) e V (azul) separadamente
vis_hv = ref_img.copy()
for rho, theta in h_lines:
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    L = max(ref_img.shape[:2])
    cv2.line(vis_hv, (int(x0 + L*(-b)), int(y0 + L*a)),
             (int(x0 - L*(-b)), int(y0 - L*a)), (0, 255, 0), 1)
for rho, theta in v_lines:
    a, b = np.cos(theta), np.sin(theta)
    x0, y0 = a * rho, b * rho
    L = max(ref_img.shape[:2])
    cv2.line(vis_hv, (int(x0 + L*(-b)), int(y0 + L*a)),
             (int(x0 - L*(-b)), int(y0 - L*a)), (255, 0, 0), 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(vis_lines, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Todas as linhas ({len(lines) if lines is not None else 0})")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(vis_hv, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Classificadas: verde=H ({len(h_lines)}), azul=V ({len(v_lines)})")
axes[1].axis("off")
plt.suptitle("Transformada de Hough — deteccao de linhas", fontsize=13)
plt.tight_layout()
save_fig("05_hough_lines", RUN_DIR)
plt.show()

## 3.4. Sobreposicao do grid na imagem original

**Por que**: Validar visualmente se os cantos anotados estao corretos — o grid 8x8 deve alinhar com as casas reais do tabuleiro.

**Conceito**: Usamos uma **homografia** que mapeia coordenadas do grid (0-8, 0-8) para pixels da imagem original. As linhas do grid sao desenhadas como curvas (nao retas) porque a projecao perspectiva faz linhas paralelas convergirem para pontos de fuga.

**Parametros**:
- `ref_corners`: 4 cantos da area de jogo (TL, TR, BR, BL)
- `color`, `thickness`: aparencia visual do grid

In [ ]:
vis_grid = draw_grid_overlay(ref_img, ref_corners, color=(0, 255, 0), thickness=2)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(cv2.cvtColor(vis_grid, cv2.COLOR_BGR2RGB))
ax.set_title("Grid 8x8 projetado sobre a imagem original")
ax.axis("off")
save_fig("06_grid_overlay", RUN_DIR)
plt.show()

## 3.5. Ajuste interativo de parametros

Use os sliders abaixo para explorar como cada parametro afeta a deteccao:
- **Imagem**: qual amostra do dataset usar
- **Canny low/high**: limiares do detector de bordas
- **Hough threshold**: minimo de votos para detectar uma linha
- **border_frac**: ajuste fino da borda do frame (0 = sem correcao)

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

def explore_grid(img_idx, canny_low, canny_high, hough_thresh, border_frac):
    # Carregar imagem
    path = images[img_idx]
    img = cv2.imread(str(path))
    ann = load_annotation(annotation_path_for(path))
    corners = order_corners(corners_to_pixels(ann["corners"], img.shape))

    # Aplicar border_frac se > 0
    if border_frac > 0:
        from src.chess import frame_to_grid_corners
        corners = frame_to_grid_corners(corners, border_frac)

    # Pre-processamento + edges + Hough
    gray = to_gray(img)
    blurred_img = blur(gray, ksize=5)
    edges_img = detect_edges(blurred_img, low=canny_low, high=canny_high)
    lines_img = detect_lines_hough(edges_img, threshold=hough_thresh)
    n_lines = len(lines_img) if lines_img is not None else 0

    # Visualizacoes
    vis_edges = cv2.cvtColor(edges_img, cv2.COLOR_GRAY2RGB)
    vis_hough = draw_hough_lines(img, lines_img, color=(0, 0, 255), thickness=1)
    vis_grid = draw_grid_overlay(img, corners, color=(0, 255, 0), thickness=2)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(vis_edges)
    axes[0].set_title(f"Canny ({canny_low}, {canny_high})")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(vis_hough, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Hough (thresh={hough_thresh}, {n_lines} linhas)")
    axes[1].axis("off")
    axes[2].imshow(cv2.cvtColor(vis_grid, cv2.COLOR_BGR2RGB))
    axes[2].set_title("Grid 8x8")
    axes[2].axis("off")
    plt.suptitle(f"Imagem {path.stem}", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(
    explore_grid,
    img_idx=IntSlider(value=0, min=0, max=min(49, len(images)-1), step=1, description="Imagem:"),
    canny_low=IntSlider(value=50, min=10, max=150, step=10, description="Canny low:"),
    canny_high=IntSlider(value=150, min=50, max=300, step=10, description="Canny high:"),
    hough_thresh=IntSlider(value=100, min=30, max=250, step=10, description="Hough thr:"),
    border_frac=FloatSlider(value=0.0, min=0.0, max=0.10, step=0.005, description="border_frac:",
                            readout_format=".3f"),
);

# 4. Correcao de Perspectiva

**Conceito**: Homografia — transformacao projetiva 2D (matriz 3x3) que mapeia 4 pontos de origem a 4 pontos de destino. `cv2.findHomography` estima a matriz; `cv2.warpPerspective` aplica a transformacao.

Mapeamos os 4 cantos do tabuleiro para os cantos de um quadrado 480x480, eliminando a distorcao de perspectiva.

In [ ]:
DST_SIZE = 480
H = compute_homography(ref_corners, dst_size=DST_SIZE)
warped = warp_board(ref_img, H, size=DST_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(cv2.cvtColor(ref_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original (perspectiva)")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Corrigido (top-down, {DST_SIZE}x{DST_SIZE})")
axes[1].axis("off")
plt.suptitle("Correcao de perspectiva via homografia", fontsize=13)
plt.tight_layout()
save_fig("07_perspective_correction", RUN_DIR)
plt.show()

# 5. Segmentacao do Grid 8x8

**Por que**: Precisamos isolar cada casa do tabuleiro para analisar individualmente se contem uma peca.

**Conceito**: Apos a correcao de perspectiva, o tabuleiro ocupa toda a imagem de `DST_SIZE x DST_SIZE` pixels como um quadrado perfeito. Dividimos em 8x8 regioes iguais de `cell_size x cell_size` pixels. Cada celula corresponde a uma casa (A1-H8).

**Parametros**:
- `grid_size=8`: divisao 8x8 (padrao de xadrez)
- `DST_SIZE=480`: resolucao da imagem retificada → cada celula tem 60x60 px

In [ ]:
cells = segment_grid(warped, grid_size=8)
print(f"Grid: {len(cells)}x{len(cells[0])}, celula: {cells[0][0].shape}")

fig, axes = plt.subplots(8, 8, figsize=(12, 12))
for row in range(8):
    for col in range(8):
        axes[row, col].imshow(cv2.cvtColor(cells[row][col], cv2.COLOR_BGR2RGB))
        axes[row, col].set_title(cell_name(row, col), fontsize=7, pad=1)
        axes[row, col].axis("off")
plt.suptitle("Grid completo 8x8 segmentado", fontsize=14)
plt.tight_layout()
save_fig("08_full_grid", RUN_DIR)
plt.show()

# 6. Deteccao de Ocupacao

Para cada celula, determinamos se ela contem uma peca usando features classicas.

## 6.1. Features classicas

| Feature | Tecnica | Logica |
|---|---|---|
| **Desvio-padrao** | Std de intensidade | Pecas aumentam variabilidade |
| **Densidade de bordas** | Proporcao de pixels Canny | Pecas geram bordas internas |
| **Variancia Laplaciano** | `cv2.Laplacian` → var | Textura/nitidez da peca |
| **Dif. centro-borda** | |centro - borda| | Peca geralmente no centro |

Classificacao: **votacao** — se >= 2 features excedem limiar, a casa e ocupada.

In [ ]:
# Extrair features de exemplos: 3 ocupadas e 3 vazias (pelo GT)
gt_occ = ground_truth_occupancy(ref_ann["config"])
occupied_coords = [(r, c) for r in range(8) for c in range(8) if gt_occ[r, c]]
empty_coords = [(r, c) for r in range(8) for c in range(8) if not gt_occ[r, c]]

examples = []
for label, coords in [("Ocupada", occupied_coords[:3]), ("Vazia", empty_coords[:3])]:
    for r, c in coords:
        feats = cell_features(cells[r][c])
        examples.append((label, cell_name(r, c), feats, cells[r][c]))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for i, (label, name, feats, cell) in enumerate(examples):
    ax = axes[i // 3, i % 3]
    ax.imshow(cv2.cvtColor(cell, cv2.COLOR_BGR2RGB))
    ax.set_title(
        f"{name} ({label})\n"
        f"std={feats['std_intensity']:.1f}  edge={feats['edge_density']:.3f}\n"
        f"tex={feats['texture_var']:.0f}  cdiff={feats['center_diff']:.1f}",
        fontsize=8,
    )
    ax.axis("off")
plt.suptitle("Features classicas: celulas ocupadas vs. vazias", fontsize=13)
plt.tight_layout()
save_fig("09_cell_features", RUN_DIR)
plt.show()

## 6.1.1. Espaco de features — ocupadas vs. vazias

Visualizando a separacao no espaco de features entre celulas com e sem pecas.
A qualidade da separacao determina a acuracia do classificador por votacao.


In [ ]:
# Coletar features de TODAS as celulas com label GT
all_feats = {"edge_density": [], "texture_var": [], "std_intensity": [], "center_diff": []}
all_labels = []

for r in range(8):
    for c in range(8):
        f = cell_features(cells[r][c])
        for k in all_feats:
            all_feats[k].append(f[k])
        all_labels.append(gt_occ[r, c])

all_labels = np.array(all_labels)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Scatter: edge_density vs texture_var
ax = axes[0, 0]
occ_mask = all_labels
ax.scatter(np.array(all_feats["edge_density"])[~occ_mask],
           np.array(all_feats["texture_var"])[~occ_mask],
           c="green", alpha=0.6, label="Vazia", s=40)
ax.scatter(np.array(all_feats["edge_density"])[occ_mask],
           np.array(all_feats["texture_var"])[occ_mask],
           c="red", alpha=0.6, label="Ocupada", s=40)
ax.axvline(0.04, color="gray", ls="--", alpha=0.5, label="threshold")
ax.axhline(80, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("edge_density")
ax.set_ylabel("texture_var")
ax.legend()
ax.set_title("Edge Density vs Texture Variance")

# Scatter: std_intensity vs center_diff
ax = axes[0, 1]
ax.scatter(np.array(all_feats["std_intensity"])[~occ_mask],
           np.array(all_feats["center_diff"])[~occ_mask],
           c="green", alpha=0.6, label="Vazia", s=40)
ax.scatter(np.array(all_feats["std_intensity"])[occ_mask],
           np.array(all_feats["center_diff"])[occ_mask],
           c="red", alpha=0.6, label="Ocupada", s=40)
ax.axvline(18, color="gray", ls="--", alpha=0.5)
ax.axhline(12, color="gray", ls="--", alpha=0.5)
ax.set_xlabel("std_intensity")
ax.set_ylabel("center_diff")
ax.legend()
ax.set_title("Std Intensity vs Center Diff")

# Histograms
for ax, feat_name, thresh in zip(axes[1], ["edge_density", "texture_var"], [0.04, 80]):
    vals = np.array(all_feats[feat_name])
    ax.hist(vals[~all_labels], bins=20, alpha=0.6, color="green", label="Vazia")
    ax.hist(vals[all_labels], bins=20, alpha=0.6, color="red", label="Ocupada")
    ax.axvline(thresh, color="black", ls="--", label=f"threshold={thresh}")
    ax.set_xlabel(feat_name)
    ax.set_ylabel("Celulas")
    ax.legend()
    ax.set_title(f"Distribuicao: {feat_name}")

plt.suptitle("Espaco de features — separacao ocupada vs. vazia", fontsize=13)
plt.tight_layout()
save_fig("09b_feature_space", RUN_DIR)
plt.show()


## 6.2. Mapa de ocupacao detectado

**O que faz**: Aplica o classificador por votacao em todas as 64 celulas e gera o mapa de ocupacao. Compara com o ground truth (anotacoes do dataset) para calcular metricas de acuracia.

**Compensacao de rotacao**: Como a camera fotografa o tabuleiro de angulos variados, o mapa detectado pode estar rotacionado 0°, 90°, 180° ou 270° em relacao ao GT. A funcao `best_rotation_vs_gt` testa as 4 opcoes e usa a melhor.

In [ ]:
occupancy, features_grid = build_occupancy_map(cells)

# Alinhar rotacao com o ground truth (o angulo da camera varia por imagem)
occupancy, rot_k = best_rotation_vs_gt(occupancy, gt_occ)
warped = np.rot90(warped, rot_k)  # rotacionar imagem junto
print(f"Rotacao aplicada: {rot_k}x90° (compensando orientacao da camera)")

occ_img = draw_occupancy_image(occupancy)
gt_img = draw_occupancy_image(gt_occ)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[0].set_title("Tabuleiro corrigido")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(occ_img, cv2.COLOR_BGR2RGB))
axes[1].set_title("Detectado (alinhado)")
axes[1].axis("off")
axes[2].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
axes[2].set_title("Ground truth")
axes[2].axis("off")
plt.suptitle("Ocupacao: detectado vs. ground truth", fontsize=13)
plt.tight_layout()
save_fig("10_occupancy_comparison", RUN_DIR)
plt.show()

# Metricas
correct = np.sum(occupancy == gt_occ)
total = gt_occ.size
accuracy = correct / total
tp = np.sum(occupancy & gt_occ)
fp = np.sum(occupancy & ~gt_occ)
fn = np.sum(~occupancy & gt_occ)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"Acuracia:  {accuracy:.1%} ({correct}/{total})")
print(f"Precision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"F1-score:  {f1:.1%}")

## 6.2.1. Ajuste interativo de thresholds de ocupacao

Use os sliders para ajustar os limiares do classificador por votacao e observe
o impacto na acuracia em tempo real.


In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def tune_thresholds(edge_t, texture_t, std_t, center_t, min_v):
    occ_tuned = np.zeros((8, 8), dtype=bool)
    for r in range(8):
        for c in range(8):
            f = cell_features(cells[r][c])
            votes = (int(f["edge_density"] > edge_t)
                     + int(f["texture_var"] > texture_t)
                     + int(f["std_intensity"] > std_t)
                     + int(f["center_diff"] > center_t))
            occ_tuned[r, c] = votes >= min_v

    occ_aligned, k = best_rotation_vs_gt(occ_tuned, gt_occ)
    acc = np.mean(occ_aligned == gt_occ)
    tp = np.sum(occ_aligned & gt_occ)
    fp = np.sum(occ_aligned & ~gt_occ)
    fn = np.sum(~occ_aligned & gt_occ)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    occ_img = draw_occupancy_image(occ_aligned)
    gt_img = draw_occupancy_image(gt_occ)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(occ_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Detectado")
    axes[0].axis("off")
    axes[1].imshow(cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")
    plt.suptitle(f"Acc={acc:.1%}  Prec={prec:.1%}  Rec={rec:.1%}  F1={f1:.1%}", fontsize=13)
    plt.tight_layout()
    plt.show()

interact(
    tune_thresholds,
    edge_t=FloatSlider(value=0.04, min=0.01, max=0.15, step=0.005, description="edge_thresh:"),
    texture_t=FloatSlider(value=80, min=20, max=300, step=10, description="texture_thr:"),
    std_t=FloatSlider(value=18, min=5, max=50, step=1, description="std_thresh:"),
    center_t=FloatSlider(value=12, min=2, max=40, step=1, description="center_thr:"),
    min_v=IntSlider(value=2, min=1, max=4, step=1, description="min_votes:"),
);


## 6.3. Analise da dificuldade do dataset

O dataset sintetico utiliza pecas e tabuleiros de **madeira com tons semelhantes**, criando baixo contraste visual entre peca e fundo. Isso torna a separacao por features classicas extremamente desafiadora.

A distribuicao abaixo mostra que as features tem grande sobreposicao entre as classes — caracteristica inerente a datasets com pecas e tabuleiro do mesmo material.

In [ ]:
occ_feats_dict = {"std_intensity": [], "edge_density": [], "texture_var": [], "center_diff": []}
emp_feats_dict = {"std_intensity": [], "edge_density": [], "texture_var": [], "center_diff": []}

for row in range(8):
    for col in range(8):
        target = occ_feats_dict if gt_occ[row, col] else emp_feats_dict
        for key in target:
            target[key].append(features_grid[row][col][key])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
feature_labels = [
    ("std_intensity", "Desvio-padrao de intensidade"),
    ("edge_density", "Densidade de bordas (Canny)"),
    ("texture_var", "Variancia do Laplaciano"),
    ("center_diff", "Diferenca centro-borda"),
]

for ax, (key, title) in zip(axes.flat, feature_labels):
    ax.hist(emp_feats_dict[key], bins=12, alpha=0.6, label="Vazia", color="green")
    ax.hist(occ_feats_dict[key], bins=12, alpha=0.6, label="Ocupada", color="red")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylabel("Contagem")

plt.suptitle("Distribuicao de features: ocupada vs. vazia (ground truth)", fontsize=13)
plt.tight_layout()
save_fig("11_feature_distributions", RUN_DIR)
plt.show()

print("Nota: a grande sobreposicao entre as distribuicoes explica a dificuldade")
print("de classificacao neste dataset especifico. Pecas e tabuleiro de madeira")
print("com tons similares criam features quase identicas para ambas as classes.")

# 6.5. Reconhecimento da cor das pecas

Alem de detectar ocupacao, tentamos classificar a **cor** de cada peca (clara ou escura)
usando o **valor (V)** no espaco de cores HSV. Pecas escuras (ebony) tem menor V que
pecas claras (boxwood).

Esta e uma demonstracao de como features no dominio de **cor** complementam as features
de **textura** e **borda** usadas na deteccao de ocupacao.


In [ ]:
from src.chess import classify_piece_color

# Classificar cor de cada peca detectada
color_map = np.full((8, 8), "", dtype="U1")
for r in range(8):
    for c in range(8):
        if occupancy[r, c]:
            color_map[r, c] = classify_piece_color(cells[r][c])

# Visualizar: branco, preto, indeterminado
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mapa de cores detectado
vis_color = np.zeros((8 * 60, 8 * 60, 3), dtype=np.uint8)
for r in range(8):
    for c in range(8):
        y1, y2 = r * 60, (r + 1) * 60
        x1, x2 = c * 60, (c + 1) * 60
        if not occupancy[r, c]:
            color = (80, 80, 80)  # vazia
        elif color_map[r, c] == "w":
            color = (200, 200, 240)  # clara
        elif color_map[r, c] == "b":
            color = (60, 40, 30)  # escura
        else:
            color = (0, 200, 200)  # indeterminada
        cv2.rectangle(vis_color, (x1+2, y1+2), (x2-2, y2-2), color, -1)
        label = cell_name(r, c)
        cv2.putText(vis_color, label, (x1+12, y1+38),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

axes[0].imshow(cv2.cvtColor(vis_color, cv2.COLOR_BGR2RGB))
axes[0].set_title("Cor detectada (clara / escura / ? / vazia)")
axes[0].axis("off")

# Ground truth: extrair cor do config
gt_color = np.full((8, 8), "", dtype="U1")
for sq, piece in ref_ann["config"].items():
    r, c = square_to_grid(sq)
    gt_color[r, c] = "w" if piece.endswith("_w") else "b"

vis_gt = np.zeros((8 * 60, 8 * 60, 3), dtype=np.uint8)
for r in range(8):
    for c in range(8):
        y1, y2 = r * 60, (r + 1) * 60
        x1, x2 = c * 60, (c + 1) * 60
        if gt_color[r, c] == "w":
            color = (200, 200, 240)
        elif gt_color[r, c] == "b":
            color = (60, 40, 30)
        else:
            color = (80, 80, 80)
        cv2.rectangle(vis_gt, (x1+2, y1+2), (x2-2, y2-2), color, -1)
        label = cell_name(r, c)
        cv2.putText(vis_gt, label, (x1+12, y1+38),
                     cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

axes[1].imshow(cv2.cvtColor(vis_gt, cv2.COLOR_BGR2RGB))
axes[1].set_title("Ground truth (cor da peca)")
axes[1].axis("off")

# Acuracia de cor (apenas celulas detectadas como ocupadas E realmente ocupadas)
correct_color = 0
total_color = 0
for r in range(8):
    for c in range(8):
        if occupancy[r, c] and gt_color[r, c] != "":
            total_color += 1
            if color_map[r, c] == gt_color[r, c]:
                correct_color += 1

plt.suptitle(f"Classificacao de cor das pecas — Acuracia: {correct_color}/{total_color}"
             f" ({correct_color/total_color:.0%})" if total_color > 0 else "Sem dados", fontsize=13)
plt.tight_layout()
save_fig("10b_piece_color_classification", RUN_DIR)
plt.show()


# 7. Deteccao de Jogadas (Analise Temporal)

**Conceito**: Comparamos os mapas de ocupacao de duas imagens para identificar casas que mudaram de estado. Em uma jogada simples, esperamos ver uma casa esvaziada e outra ocupada.

Para demonstrar a tecnica de forma robusta, usamos o **ground truth** de ocupacao — isolando assim a avaliacao da deteccao de mudancas da dificuldade de classificacao individual.

In [ ]:
if len(images) >= 2:
    img_a_path, img_b_path = images[0], images[1]

    img_a = cv2.imread(str(img_a_path))
    ann_a = load_annotation(annotation_path_for(img_a_path))
    corners_a = order_corners(corners_to_pixels(ann_a["corners"], img_a.shape))
    H_a = compute_homography(corners_a, dst_size=DST_SIZE)
    warped_a = warp_board(img_a, H_a, size=DST_SIZE)
    gt_a = ground_truth_occupancy(ann_a["config"])
    _, rot_a = best_rotation_vs_gt(np.zeros((8,8), dtype=bool), gt_a)  # dummy occ
    # Usar detect_board_rotation para rotacionar a imagem (sem precisar de GT da ocupacao)
    rot_a = detect_board_rotation(warped_a)
    warped_a = np.rot90(warped_a, rot_a)

    img_b = cv2.imread(str(img_b_path))
    ann_b = load_annotation(annotation_path_for(img_b_path))
    corners_b = order_corners(corners_to_pixels(ann_b["corners"], img_b.shape))
    H_b = compute_homography(corners_b, dst_size=DST_SIZE)
    warped_b = warp_board(img_b, H_b, size=DST_SIZE)
    gt_b = ground_truth_occupancy(ann_b["config"])
    rot_b = detect_board_rotation(warped_b)
    warped_b = np.rot90(warped_b, rot_b)

    # Rotacionar GT para alinhar com a imagem rotacionada
    gt_a_rot = np.rot90(gt_a, 4 - rot_a) if rot_a else gt_a
    gt_b_rot = np.rot90(gt_b, 4 - rot_b) if rot_b else gt_b

    changes = detect_moves(gt_a, gt_b)
    print(f"Casas que mudaram de estado: {len(changes)}")
    for r, c, change_type in changes[:10]:
        print(f"  {cell_name(r, c)}: {change_type}")
    if len(changes) > 10:
        print(f"  ... e mais {len(changes) - 10}")
else:
    print("Dataset com menos de 2 imagens.")

## 7.2. Visualizacao das diferencas

**O que faz**: Renderiza os mapas de ocupacao dos dois frames lado a lado e destaca as casas que mudaram de estado.

**Convencao de cores**:
- **Amarelo**: casa esvaziou (peca saiu)
- **Ciano**: casa ocupou (peca chegou)
- **Cinza**: sem mudanca

In [ ]:
if len(images) >= 2:
    occ_img_a = draw_occupancy_image(gt_a)
    occ_img_b = draw_occupancy_image(gt_b)
    diff_img = draw_move_diff(gt_a, gt_b)

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    axes[0, 0].imshow(cv2.cvtColor(warped_a, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title(f"Frame A ({img_a_path.stem})")
    axes[0, 0].axis("off")
    axes[0, 1].imshow(cv2.cvtColor(warped_b, cv2.COLOR_BGR2RGB))
    axes[0, 1].set_title(f"Frame B ({img_b_path.stem})")
    axes[0, 1].axis("off")
    axes[0, 2].axis("off")
    axes[0, 2].text(0.5, 0.5, f"{len(changes)} mudancas\ndetectadas",
                     ha="center", va="center", fontsize=16,
                     transform=axes[0, 2].transAxes)

    axes[1, 0].imshow(cv2.cvtColor(occ_img_a, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title("Ocupacao A (GT)")
    axes[1, 0].axis("off")
    axes[1, 1].imshow(cv2.cvtColor(occ_img_b, cv2.COLOR_BGR2RGB))
    axes[1, 1].set_title("Ocupacao B (GT)")
    axes[1, 1].axis("off")
    axes[1, 2].imshow(cv2.cvtColor(diff_img, cv2.COLOR_BGR2RGB))
    axes[1, 2].set_title("Diferencas (amarelo=esvaziou, ciano=ocupou)")
    axes[1, 2].axis("off")

    plt.suptitle("Deteccao de jogadas — analise temporal", fontsize=14)
    plt.tight_layout()
    save_fig("12_move_detection", RUN_DIR)
    plt.show()

# 8. Pipeline Completa em Multiplas Imagens

**Por que**: Avaliar a robustez do pipeline em diferentes imagens, com diferentes posicoes de pecas e angulos de camera.

**O que faz**: Executa a pipeline completa (cantos → homografia → warp → segmentacao → features → classificacao) nas primeiras `N_EVAL` imagens e computa metricas agregadas (acuracia, precisao, recall, F1).

**Parametros**:
- `N_EVAL=10`: numero de imagens para avaliacao

In [ ]:
N_EVAL = min(10, len(images))
results = []

for i in range(N_EVAL):
    img_path = images[i]
    img = cv2.imread(str(img_path))
    ann = load_annotation(annotation_path_for(img_path))
    gt = ground_truth_occupancy(ann["config"])
    corners = order_corners(corners_to_pixels(ann["corners"], img.shape))
    H = compute_homography(corners, dst_size=DST_SIZE)
    w = warp_board(img, H, size=DST_SIZE)
    c = segment_grid(w)
    occ, _ = build_occupancy_map(c)

    # Alinhar rotacao com GT (camera varia por imagem)
    occ, _ = best_rotation_vs_gt(occ, gt)

    acc = np.mean(occ == gt)
    tp = np.sum(occ & gt)
    fp = np.sum(occ & ~gt)
    fn = np.sum(~occ & gt)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    results.append({
        "image": img_path.stem, "accuracy": float(acc),
        "precision": float(prec), "recall": float(rec), "f1": float(f1),
        "n_pieces_gt": int(gt.sum()), "n_pieces_det": int(occ.sum()),
    })
    print(f"{img_path.stem}: acc={acc:.1%}  f1={f1:.1%}  (GT={int(gt.sum())}, det={int(occ.sum())})")

mean_acc = np.mean([r["accuracy"] for r in results])
mean_f1 = np.mean([r["f1"] for r in results])
print(f"\nMedia: acc={mean_acc:.1%}, f1={mean_f1:.1%}")

## 8.1. Grafico de desempenho por imagem

Visualizacao da acuracia e F1-score para cada imagem avaliada.
A linha tracejada indica a media. Imagens com pecas em posicoes
mais complexas (muitas pecas proximas, sombras fortes) tendem a ter menor acuracia.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
names = [r["image"] for r in results]
accs = [r["accuracy"] for r in results]
f1s = [r["f1"] for r in results]
x = np.arange(len(names))
width = 0.35
ax.bar(x - width/2, accs, width, label="Acuracia", color="steelblue")
ax.bar(x + width/2, f1s, width, label="F1-score", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.legend()
ax.axhline(y=mean_acc, color="steelblue", linestyle="--", alpha=0.5)
ax.axhline(y=mean_f1, color="coral", linestyle="--", alpha=0.5)
ax.set_title("Metricas de ocupacao por imagem")
plt.tight_layout()
save_fig("13_metrics_bar", RUN_DIR)
plt.show()

# 9. Estado Atual e Proximos Passos

Este projeto esta em desenvolvimento. O objetivo final e construir um sistema completo que:

| Etapa | Descricao | Status |
| --- | --- | --- |
| **1. Leitura do tabuleiro** | Detectar o tabuleiro, corrigir perspectiva, segmentar em 8x8 | Concluido |
| **2. Identificacao de ocupacao** | Determinar quais casas tem pecas (ocupada/vazia) | Concluido |
| **3. Classificacao de pecas** | Identificar o tipo de cada peca (peao, torre, bispo, etc.) | Em andamento |
| **4. Indicacao de jogadas** | Comparar estados, sugerir/validar lances, gerar notacao PGN | Planejado |

## O que ja funciona (Etapas 1-2)

- Pre-processamento: tons de cinza, suavizacao gaussiana, equalizacao (CLAHE)
- Deteccao de bordas: Roberts, Prewitt, Sobel, Canny + operacoes morfologicas
- Deteccao de linhas via Transformada de Hough
- Correcao de perspectiva via homografia
- Segmentacao do grid 8x8
- Deteccao de ocupacao por votacao de features classicas
- Compensacao automatica de rotacao do tabuleiro
- Classificacao basica de cor das pecas (clara/escura via HSV)
- Deteccao de mudancas entre dois frames

## Proximo objetivo: Classificacao de pecas (Etapa 3)

A classificacao de cor (clara/escura) ja esta implementada. O proximo passo e identificar o **tipo** de peca em cada casa ocupada. Abordagens classicas possiveis:

- **Template matching** (NCC/SSD): comparar cada celula com templates de referencia de cada tipo de peca
- **Hu Moments**: descritores de forma invariantes a escala e rotacao
- **Contornos + area/perimetro**: pecas altas (torre, rainha) vs baixas (peao) tem silhuetas distintas
- **HOG (Histogram of Oriented Gradients)**: descritores de textura direcional

## Visao futura: Indicacao de jogadas (Etapa 4)

Com a classificacao de pecas resolvida, o sistema podera:

- Reconstruir a posicao completa (FEN notation)
- Comparar dois frames para detectar qual peca moveu e para onde
- Gerar notacao algebrica (ex: Nf3, exd5)
- Validar legalidade do lance pelas regras do xadrez


# 10. Limitacoes Conhecidas

- **Material uniforme**: pecas e tabuleiro de madeira criam baixo contraste peca-fundo — principal desafio para metodos classicos
- **Projecao 3D das pecas**: pecas altas "vazam" para celulas vizinhas na imagem retificada (limitacao inerente da homografia 2D)
- **Rotacao do tabuleiro**: o padrao xadrez tem simetria de 180, entao a deteccao automatica de orientacao so resolve 2 das 4 rotacoes possiveis
- **Limiares fixos**: os thresholds de ocupacao foram calibrados para o dataset sintetico e podem nao generalizar para imagens reais
- **Sem identificacao de tipo de peca**: atualmente so detecta ocupacao e cor, nao o tipo (peao, torre, etc.)


# 11. Salvando Metricas

**O que faz**: Exporta as metricas de avaliacao em formato JSON para o diretorio `outputs/results/`, permitindo comparacao entre diferentes execucoes e parametrizacoes.

In [ ]:
from src.utils import save_metrics

metrics = {
    "n_images_evaluated": len(results),
    "mean_accuracy": float(mean_acc),
    "mean_f1": float(mean_f1),
    "per_image": results,
}
save_metrics(metrics, RUN_DIR)
print("Metricas salvas.")